# Pandas and SQLAlchemy (2.0 style)

Pandas has several functions to work with databases via SQLAlchemy.

This can be an easier way to get data from a database and to pass data into a database.

This notebook uses SQLAlchemy 2.x Core patterns and requires pandas 2.x or later.

Let's start with loading the `world.sqlite` database we've been using.

In [ ]:
#!/usr/bin/env python
from sqlalchemy import create_engine, MetaData, Table, select

# Create an Engine object which is our handle into the database.
engine = create_engine('sqlite:////blue/bsc4452/share/Class_Files/data/world.sqlite')

# MetaData holds the table definitions. Tables are reflected from the existing database below.
metadata = MetaData()

The cell below is also from the [SQLAlchemy 2.0 notebook](SQLAlchemy_2.0.ipynb). It creates the `city` object by reflecting the table from the database (using `autoload_with=engine`) and builds the `query` variable with `select()`. It then shows how you would execute the query and print the rows, using a connection in a `with` block so it is closed automatically.

In [ ]:
# Auto-create the city object by reflecting the table from the database.
city = Table('city', metadata, autoload_with=engine)

# Mirroring this command: SELECT Id, Name, Population FROM city LIMIT 10;
query = select(city.c.ID, city.c.Name, city.c.Population).limit(10)

with engine.connect() as conn:
    result = conn.execute(query)
    for row in result:
        print(row)

print(type(query))

## Reading SQL query results into a Pandas data frame

Here's how we can put the result into a pandas data frame. The selectable and a connection (or engine) are the required arguments for `read_sql`. In this example, we also set the `ID` column to be used as the data frame index.

In [ ]:
import pandas as pd

with engine.connect() as conn:
    df = pd.read_sql(query, conn, index_col='ID')

df.head()

## Writing a Pandas data frame to a database

In the next example, we'll use the `shipping_tables.xlsx` table that we used in the [Reading poorly structured Excel files with Pandas notebook](Pandas_messy_data.ipynb).

First, we'll get the data into data frames.

In [ ]:
orders = pd.read_excel('shipping_tables.xlsx', header=1, usecols='B:F')
orders.head()

In [ ]:
from openpyxl import load_workbook

wb = load_workbook(filename='shipping_tables.xlsx')
sheet = wb['shipping_rates']
lookup_table = sheet.tables['ship_cost']
data = sheet[lookup_table.ref]
rows_list = []
for row in data:
    cols = []
    for col in row:
        cols.append(col.value)
    rows_list.append(cols)
ship_rates = pd.DataFrame(data=rows_list[1:], index=None, columns=rows_list[0])
ship_rates.head()

Now we have two Pandas data frames and can create a database and add the data.

In SQLAlchemy 2.x, writes should happen inside a transaction. `engine.begin()` opens a connection and commits automatically when the block finishes (or rolls back if there is an error).

In [ ]:
# Create the database engine.
shipping_engine = create_engine('sqlite:///shipping.sqlite')

# Use the `to_sql` function to send the data frames to the database.
# engine.begin() commits the transaction automatically.
with shipping_engine.begin() as conn:
    orders.to_sql('orders', conn, if_exists='replace')
    ship_rates.to_sql('ship_rates', conn, if_exists='replace')

Finally, run a raw SQL query to see that the data are there. In SQLAlchemy 2.x, `engine.execute()` has been removed, so we use a connection with `exec_driver_sql()` (or `text()`). We could create objects, read the metadata, etc. but that seems excessive here.

In [ ]:
with shipping_engine.connect() as conn:
    rows = conn.exec_driver_sql("SELECT * FROM orders").fetchall()

rows